# GitHub仓库三维关联度分析

本Notebook实现了基于生态、语义、时序三个维度的GitHub仓库关联度分析算法。

## 算法概述

```
总体关联度 = 0.4 × 生态关联度 + 0.4 × 语义关联度 + 0.2 × 时序关联度
```

## 1. 环境配置与导入库

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import defaultdict
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("Libraries imported successfully!")

## 2. 数据加载

In [ ]:

with open('all.json', 'r', encoding='utf-8') as f:
    repos_data = json.load(f)

print(f"Total repositories: {len(repos_data)}")

## 3. 数据预处理与特征提取

In [ ]:
def clean_text(text):

    if not text:
        return ""
    text = re.sub(r'<[^>]+>', '', str(text))
    text = re.sub(r'\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()


features_list = []
for repo in repos_data:
    basic_info = repo.get('basic_info', {})
    feature = {
        'repo_name': basic_info.get('full_name', ''),
        'language': basic_info.get('language', ''),
        'description': clean_text(basic_info.get('description', '')),
        'topics': basic_info.get('topics', []),
        'star_count': basic_info.get('stargazers_count', 0),
        'contributor_network': repo.get('contributor_network', []),
        'stars_timeline': repo.get('stars_timeline', []),
        'readme_content': clean_text(repo.get('readme_content', '')[:3000])
    }
    features_list.append(feature)

df = pd.DataFrame(features_list)
print(f"Features extracted: {df.shape[1]} columns")
df.head()

## 4. 数据完整性检查

In [ ]:
print("=== Data Completeness Check ===")
print(f"contributor_network available: {df['contributor_network'].apply(lambda x: len(x) > 0).sum()}/{len(df)}")
print(f"stars_timeline available: {df['stars_timeline'].apply(lambda x: len(x) > 0).sum()}/{len(df)}")
print(f"description available: {df['description'].apply(lambda x: len(x) > 0).sum()}/{len(df)}")
print(f"topics available: {df['topics'].apply(lambda x: len(x) > 0).sum()}/{len(df)}")
print(f"readme_content available: {df['readme_content'].apply(lambda x: len(x) > 0).sum()}/{len(df)}")

## 5. 组合文本特征（用于语义分析）

In [ ]:
def combine_text_features(row):
    parts = []
    
    if row['description']:
        parts.append(row['description'])
    
    if row['topics']:
        parts.append(' '.join(row['topics']))
    
    if row['language']:
        parts.append(row['language'])
    
    if row['readme_content']:
        parts.append(row['readme_content'])
    
    return ' '.join(parts)

df['combined_text'] = df.apply(combine_text_features, axis=1)
print("Text features combined")

## 6. 计算生态关联度 (Ecological Similarity)

In [ ]:
def calculate_ecological_similarity_optimized(df):
    n = len(df)
    eco_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                eco_matrix[i, j] = 1.0
            else:
                contributors_i = set([c['login'] for c in df.iloc[i]['contributor_network']])
                contributors_j = set([c['login'] for c in df.iloc[j]['contributor_network']])
                
                if len(contributors_i) > 0 and len(contributors_j) > 0:
                    intersection = len(contributors_i & contributors_j)
                    union = len(contributors_i | contributors_j)
                    contributor_sim = intersection / union
                else:
                    contributor_sim = 0.0
                
                lang_i = df.iloc[i]['language'].lower()
                lang_j = df.iloc[j]['language'].lower()
                language_sim = 1.0 if lang_i == lang_j and lang_i != '' else 0.0
                
                topics_i = set(df.iloc[i]['topics']) if df.iloc[i]['topics'] else set()
                topics_j = set(df.iloc[j]['topics']) if df.iloc[j]['topics'] else set()
                
                if len(topics_i) > 0 and len(topics_j) > 0:
                    intersection = len(topics_i & topics_j)
                    union = len(topics_i | topics_j)
                    topic_sim = intersection / union
                else:
                    topic_sim = 0.0
                
                star_i = df.iloc[i]['star_count']
                star_j = df.iloc[j]['star_count']
                max_star = max(star_i, star_j)
                min_star = min(star_i, star_j)
                
                if max_star > 0:
                    star_sim = min_star / max_star
                else:
                    star_sim = 1.0 if star_i == star_j else 0.0
                
                eco_matrix[i, j] = (
                    0.4 * contributor_sim +
                    0.3 * language_sim +
                    0.2 * topic_sim +
                    0.1 * star_sim
                )
    
    return eco_matrix

print("Calculating optimized ecological similarity matrix...")
eco_matrix = calculate_ecological_similarity_optimized(df)
print(f"Ecological similarity matrix shape: {eco_matrix.shape}")
print(f"Mean: {eco_matrix.mean():.4f}, Std: {eco_matrix.std():.4f}")
print(f"Min: {eco_matrix.min():.4f}, Max: {eco_matrix.max():.4f}")

## 7. 计算语义关联度 (Semantic Similarity)

In [ ]:
def calculate_semantic_similarity_optimized(df):
    texts = df['combined_text'].tolist()
    
    vectorizer = TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.95,
        stop_words='english',
        sublinear_tf=True
    )
    
    tfidf_matrix = vectorizer.fit_transform(texts)
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    n = len(df)
    language_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            lang_i = df.iloc[i]['language'].lower()
            lang_j = df.iloc[j]['language'].lower()
            if lang_i == lang_j and lang_i != '':
                language_matrix[i, j] = 1.0
    
    combined_matrix = 0.8 * similarity_matrix + 0.2 * language_matrix
    
    np.fill_diagonal(combined_matrix, 1.0)
    
    return combined_matrix

print("Calculating optimized semantic similarity matrix...")
semantic_matrix = calculate_semantic_similarity_optimized(df)
print(f"Semantic similarity matrix shape: {semantic_matrix.shape}")
print(f"Mean: {semantic_matrix.mean():.4f}, Std: {semantic_matrix.std():.4f}")
print(f"Min: {semantic_matrix.min():.4f}, Max: {semantic_matrix.max():.4f}")

## 8. 计算时序关联度 (Temporal Similarity)

In [ ]:
def extract_stars_growth_rate(stars_timeline):
    if not stars_timeline or len(stars_timeline) < 2:
        return np.zeros(23)
    
    sorted_timeline = sorted(stars_timeline, key=lambda x: x['month'])
    stars_values = [item['stars'] for item in sorted_timeline]
    
    growth_rates = []
    for i in range(1, len(stars_values)):
        if stars_values[i-1] > 0:
            growth_rate = (stars_values[i] - stars_values[i-1]) / stars_values[i-1]
        else:
            growth_rate = 0
        growth_rates.append(growth_rate)
    
    return np.array(growth_rates[:23])

def calculate_dtw_distance(series_a, series_b):
    n, m = len(series_a), len(series_b)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(series_a[i - 1] - series_b[j - 1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i - 1, j],
                dtw_matrix[i, j - 1],
                dtw_matrix[i - 1, j - 1]
            )
    
    return dtw_matrix[n, m]

def calculate_temporal_similarity_optimized(df):

    n = len(df)
    temporal_matrix = np.zeros((n, n))
    
    growth_rates_list = [extract_stars_growth_rate(repo['stars_timeline']) for repo in df.to_dict('records')]
    
    max_global_dist = 0
    for i in range(n):
        for j in range(i + 1, n):
            dist = calculate_dtw_distance(growth_rates_list[i], growth_rates_list[j])
            if dist > max_global_dist:
                max_global_dist = dist
    
    print(f"Global max DTW distance: {max_global_dist:.2f}")
    
    for i in range(n):
        for j in range(n):
            if i == j:
                temporal_matrix[i, j] = 1.0
            else:
                dtw_dist = calculate_dtw_distance(growth_rates_list[i], growth_rates_list[j])
                
                if max_global_dist == 0:
                    temporal_matrix[i, j] = 1.0
                else:
                    temporal_matrix[i, j] = max(0, 1 - dtw_dist / max_global_dist)
    
    return temporal_matrix

print("Calculating optimized temporal similarity matrix...")
temporal_matrix = calculate_temporal_similarity_optimized(df)
print(f"Temporal similarity matrix shape: {temporal_matrix.shape}")
print(f"Mean: {temporal_matrix.mean():.4f}, Std: {temporal_matrix.std():.4f}")
print(f"Min: {temporal_matrix.min():.4f}, Max: {temporal_matrix.max():.4f}")

## 9. 计算总体关联度

In [ ]:

total_matrix = (
    0.4 * eco_matrix +
    0.4 * semantic_matrix +
    0.2 * temporal_matrix
)

np.fill_diagonal(total_matrix, 1.0)

print(f"Total similarity matrix shape: {total_matrix.shape}")
print(f"Mean: {total_matrix.mean():.4f}, Std: {total_matrix.std():.4f}")
print(f"Min: {total_matrix.min():.4f}, Max: {total_matrix.max():.4f}")

## 10. 保存结果

In [ ]:

np.save('repo_similarity_matrix_722_optimized.npy', total_matrix)
print("Similarity matrix saved: repo_similarity_matrix_722_optimized.npy")


similarity_df = pd.DataFrame(total_matrix, index=df['repo_name'], columns=df['repo_name'])
similarity_df.to_csv('repo_similarity_matrix_722_optimized.csv')
print("Similarity matrix CSV saved: repo_similarity_matrix_722_optimized.csv")


df.to_csv('github_repos_features_722_optimized.csv', index=False)
print("Features CSV saved: github_repos_features_722_optimized.csv")

## 11. 查找最相似的仓库对

In [ ]:

similar_pairs = []
n = len(df)

for i in range(n):
    for j in range(i + 1, n):
        similar_pairs.append({
            'repo1': df.iloc[i]['repo_name'],
            'repo2': df.iloc[j]['repo_name'],
            'eco_score': eco_matrix[i, j],
            'semantic_score': semantic_matrix[i, j],
            'temporal_score': temporal_matrix[i, j],
            'total_score': total_matrix[i, j]
        })

similar_pairs_sorted = sorted(similar_pairs, key=lambda x: x['total_score'], reverse=True)
top_20_pairs = similar_pairs_sorted[:20]

top_20_df = pd.DataFrame(top_20_pairs)
print("\nTop 20 most similar repository pairs:")
print(top_20_df.to_string(index=False))

## 12. 生成完整结果JSON

In [ ]:
result = {
    'metadata': {
        'total_repos': len(df),
        'calculation_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'algorithm_version': '2.0'
    },
    'statistics': {
        'ecological': {
            'mean': float(eco_matrix.mean()),
            'std': float(eco_matrix.std()),
            'min': float(eco_matrix.min()),
            'max': float(eco_matrix.max())
        },
        'semantic': {
            'mean': float(semantic_matrix.mean()),
            'std': float(semantic_matrix.std()),
            'min': float(semantic_matrix.min()),
            'max': float(semantic_matrix.max())
        },
        'temporal': {
            'mean': float(temporal_matrix.mean()),
            'std': float(temporal_matrix.std()),
            'min': float(temporal_matrix.min()),
            'max': float(temporal_matrix.max())
        },
        'total': {
            'mean': float(total_matrix.mean()),
            'std': float(total_matrix.std()),
            'min': float(total_matrix.min()),
            'max': float(total_matrix.max())
        }
    },
    'top_similar_pairs': top_20_pairs,
    'repositories': df['repo_name'].tolist()
}

with open('repo_similarity_result_722_optimized.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("\nFinal result saved: repo_similarity_result_722_optimized.json")

## 13. 数据可视化

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))


axes[0, 0].hist(eco_matrix[np.triu_indices_from(eco_matrix, k=1)], bins=50, alpha=0.7, color='blue')
axes[0, 0].set_title('Ecological Similarity Distribution', fontsize=12)
axes[0, 0].set_xlabel('Similarity Score')
axes[0, 0].set_ylabel('Frequency')


axes[0, 1].hist(semantic_matrix[np.triu_indices_from(semantic_matrix, k=1)], bins=50, alpha=0.7, color='green')
axes[0, 1].set_title('Semantic Similarity Distribution', fontsize=12)
axes[0, 1].set_xlabel('Similarity Score')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(temporal_matrix[np.triu_indices_from(temporal_matrix, k=1)], bins=50, alpha=0.7, color='orange')
axes[1, 0].set_title('Temporal Similarity Distribution', fontsize=12)
axes[1, 0].set_xlabel('Similarity Score')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(total_matrix[np.triu_indices_from(total_matrix, k=1)], bins=50, alpha=0.7, color='red')
axes[1, 1].set_title('Total Similarity Distribution', fontsize=12)
axes[1, 1].set_xlabel('Similarity Score')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('similarity_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Similarity distributions saved: similarity_distributions.png")

In [ ]:

top_50_repos = df['repo_name'].tolist()[:50]
top_50_matrix = similarity_df.loc[top_50_repos, top_50_repos].values

plt.figure(figsize=(20, 18))
sns.heatmap(top_50_matrix, 
            xticklabels=[name.split('/')[-1][:15] for name in top_50_repos],
            yticklabels=[name.split('/')[-1][:15] for name in top_50_repos],
            cmap='YlOrRd', 
            cbar_kws={'label': 'Similarity Score'},
            fmt='.2f')
plt.title('Top 50 Repositories Similarity Heatmap', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('top_50_similarity_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Top 50 heatmap saved: top_50_similarity_heatmap.png")

In [ ]:

threshold = 0.5
G = nx.Graph()

for i, repo1 in enumerate(df['repo_name']):
    for j, repo2 in enumerate(df['repo_name']):
        if i < j and total_matrix[i, j] >= threshold:
            G.add_edge(repo1, repo2, weight=total_matrix[i, j])

print(f"Network created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

plt.figure(figsize=(25, 25))
pos = nx.spring_layout(G, k=0.3, iterations=50, seed=42)

degrees = dict(G.degree())
node_sizes = [degrees[node] * 100 for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', alpha=0.7)
nx.draw_networkx_edges(G, pos, width=0.5, alpha=0.5, edge_color='gray')
nx.draw_networkx_labels(G, pos, 
                        labels={node: node.split('/')[-1][:10] for node in G.nodes()},
                        font_size=8)

plt.title(f'GitHub Repository Similarity Network (threshold >= {threshold})', 
          fontsize=16, pad=20)
plt.axis('off')
plt.tight_layout()
plt.savefig('repo_similarity_network.png', dpi=300, bbox_inches='tight')
plt.show()
print("Network visualization saved: repo_similarity_network.png")

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8))

eco_upper = eco_matrix[np.triu_indices_from(eco_matrix, k=1)]
semantic_upper = semantic_matrix[np.triu_indices_from(semantic_matrix, k=1)]
temporal_upper = temporal_matrix[np.triu_indices_from(temporal_matrix, k=1)]
total_upper = total_matrix[np.triu_indices_from(total_matrix, k=1)]

data_to_plot = [eco_upper, semantic_upper, temporal_upper, total_upper]
labels = ['Ecological', 'Semantic', 'Temporal', 'Total']

bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)

colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_title('Three-Dimensional Similarity Comparison', fontsize=14, pad=20)
ax.set_ylabel('Similarity Score', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('similarity_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print("Boxplot saved: similarity_boxplot.png")

## 14. 查询特定仓库的相似仓库

In [ ]:
def find_similar_repos(repo_name, top_k=10):

    if repo_name not in df['repo_name'].values:
        print(f"Repository '{repo_name}' not found!")
        return None
    
    idx = df[df['repo_name'] == repo_name].index[0]
    
    similarities = []
    for j, other_repo in enumerate(df['repo_name']):
        if idx != j:
            similarities.append({
                'repo': other_repo,
                'eco_score': eco_matrix[idx, j],
                'semantic_score': semantic_matrix[idx, j],
                'temporal_score': temporal_matrix[idx, j],
                'total_score': total_matrix[idx, j]
            })
    
    similarities_sorted = sorted(similarities, key=lambda x: x['total_score'], reverse=True)
    return similarities_sorted[:top_k]


target_repo = "flutter/flutter"
similar_repos = find_similar_repos(target_repo, top_k=10)

if similar_repos:
    print(f"\nTop 10 repositories similar to '{target_repo}':")
    print("=" * 120)
    print(f"{'Rank':<6} {'Repository':<50} {'Eco':<8} {'Semantic':<10} {'Temporal':<10} {'Total':<8}")
    print("=" * 120)
    for i, repo in enumerate(similar_repos, 1):
        print(f"{i:<6} {repo['repo']:<50} {repo['eco_score']:.4f}  {repo['semantic_score']:.4f}  {repo['temporal_score']:.4f}  {repo['total_score']:.4f}")

## 15. 分析结果总结

In [ ]:
print("\n" + "="*80)
print("GitHub Repository Three-Dimensional Similarity Analysis Summary")
print("="*80)

print(f"\nTotal Repositories Analyzed: {len(df)}")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Algorithm Version: 2.0 (Optimized)")

print("\n" + "-"*80)
print("Similarity Statistics:")
print("-"*80)
print(f"{'Dimension':<15} {'Mean':<10} {'Std Dev':<10} {'Min':<10} {'Max':<10}")
print("-"*80)
print(f"{'Ecological':<15} {eco_matrix.mean():.4f}    {eco_matrix.std():.4f}    {eco_matrix.min():.4f}    {eco_matrix.max():.4f}")
print(f"{'Semantic':<15} {semantic_matrix.mean():.4f}    {semantic_matrix.std():.4f}    {semantic_matrix.min():.4f}    {semantic_matrix.max():.4f}")
print(f"{'Temporal':<15} {temporal_matrix.mean():.4f}    {temporal_matrix.std():.4f}    {temporal_matrix.min():.4f}    {temporal_matrix.max():.4f}")
print(f"{'Total':<15} {total_matrix.mean():.4f}    {total_matrix.std():.4f}    {total_matrix.min():.4f}    {total_matrix.max():.4f}")

print("\n" + "-"*80)
print("Top 5 Most Similar Repository Pairs:")
print("-"*80)
for i, pair in enumerate(top_20_pairs[:5], 1):
    print(f"\n{i}. {pair['repo1']} <-> {pair['repo2']}")
    print(f"   Total Score: {pair['total_score']:.4f}")
    print(f"   (Eco: {pair['eco_score']:.4f}, Semantic: {pair['semantic_score']:.4f}, Temporal: {pair['temporal_score']:.4f})")

print("\n" + "-"*80)
print("Generated Files:")
print("-"*80)
print("1. repo_similarity_matrix_722_optimized.npy - Similarity matrix (numpy format)")
print("2. repo_similarity_matrix_722_optimized.csv - Similarity matrix (CSV format)")
print("3. github_repos_features_722_optimized.csv - Repository features")
print("4. repo_similarity_result_722_optimized.json - Complete result with statistics")
print("5. similarity_distributions.png - Similarity distribution histograms")
print("6. top_50_similarity_heatmap.png - Top 50 repositories heatmap")
print("7. repo_similarity_network.png - Repository similarity network")
print("8. similarity_boxplot.png - Three-dimensional similarity comparison")

print("\n" + "="*80)
print("Analysis Complete!")
print("="*80)